In [ ]:
import math
import random
import numpy as np
import pandas as pd
import cv2
import os
from scipy.spatial import cKDTree
from datetime import datetime, timezone

# =======================================================
# 1. 初始化设置与参数
# =======================================================
random.seed(42)
output_dir = "star_tracker_dataset"
os.makedirs(output_dir, exist_ok=True)

# 机械误差参数
DELTA = random.uniform(-0.01, 0.01)
PHI_X, PHI_Y, PHI_Z = random.uniform(-0.02, 0.02), random.uniform(-0.02, 0.02), random.uniform(-0.02, 0.02)
THETA_NP = random.uniform(-0.005, 0.005)
EPS_X, EPS_Y, EPS_Z = random.uniform(-0.01, 0.01), random.uniform(-0.01, 0.01), random.uniform(-0.01, 0.01)
LAT, LON = 31.22, 121.48

# 相机参数 (使用 50mm 广角保证星星数量足够)
focal_length = 50.0 
pixel_size = 5.0e-3
img_width, img_height = 1920, 1080 
f_px = focal_length / pixel_size
K = np.array([[f_px, 0, img_width / 2.0], [0, f_px, img_height / 2.0], [0, 0, 1]])

# =======================================================
# 2. 载入星表并进行【硬件极限定制】
# =======================================================
print("正在载入全天区星表...")
df = pd.read_csv('gaia_northern_12mag.csv')

# ！！！核心修改：直接在源头截断，模拟星敏感器 6.5 等感光极限 ！！！
df_filtered = df[df['phot_g_mean_mag'] <= 6.5]
print(f"原数据库恒星数: {len(df)}，过滤后 (<= 6.5等) 剩余极亮恒星数: {len(df_filtered)}")

mags_all = df_filtered['phot_g_mean_mag'].values
ra_rad_all = np.radians(df_filtered['ra'].values)
dec_rad_all = np.radians(df_filtered['dec'].values)
stars_3d_all = np.vstack((np.cos(dec_rad_all)*np.cos(ra_rad_all), np.cos(dec_rad_all)*np.sin(ra_rad_all), np.sin(dec_rad_all))).T 
tree = cKDTree(stars_3d_all)

# 计算 kd-tree 搜索半径
actual_fov = np.degrees(2 * np.arctan((img_width * pixel_size) / (2 * focal_length)))
search_radius = 2.0 * np.sin(np.radians(actual_fov * 1.2) / 2.0)

# =======================================================
# 3. 基础物理与变换矩阵 (修复版)
# =======================================================
def rot_x(a): c,s=math.cos(a),math.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])
def rot_y(a): c,s=math.cos(a),math.sin(a); return np.array([[c,0,s],[0,1,0],[-s,0,c]]) # 已修复
def rot_z(a): c,s=math.cos(a),math.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])
def euler_zxy(z, x, y): return rot_x(x) @ rot_y(y) @ rot_z(z)

def julian_day(date):
    y, m, d = date.year, date.month, date.day + date.hour/24.0 + date.minute/1440.0 + date.second/86400.0
    if m <= 2: y -= 1; m += 12
    A = y // 100
    return math.floor(365.25*(y + 4716)) + math.floor(30.6001*(m + 1)) + d + (2 - A + A // 4) - 1524.5

def gmst_rad(jd):
    T = (jd - 2451545.0) / 36525.0
    return ((24110.54841 + 8640184.812866 * T + 0.093104 * T*T - 0.0000062 * T*T*T) % 86400) * 2 * math.pi / 86400

def draw_gaussian_spot(img, cx, cy, brightness, sigma=1.5):
    h, w = img.shape
    s = int(sigma * 6) | 1
    x, y = np.arange(0, s, 1, float), np.arange(0, s, 1, float)[:, np.newaxis]
    c = s // 2
    dx, dy = cx - int(cx), cy - int(cy)
    g = brightness * np.exp(-((x - c - dx)**2 + (y - c - dy)**2) / (2 * sigma**2))
    ix, iy = int(cx), int(cy)
    x1, x2 = max(0, ix - c), min(w, ix - c + s)
    y1, y2 = max(0, iy - c), min(h, iy - c + s)
    gx1, gx2 = c - (ix - x1), c + (x2 - ix)
    gy1, gy2 = c - (iy - y1), c + (y2 - iy)
    if x1 < x2 and y1 < y2: img[y1:y2, x1:x2] += g[gy1:gy2, gx1:gx2]

# =======================================================
# 4. 自动化批量生成流程
# =======================================================
utc_now = datetime.now(timezone.utc)
num_images = 20
success_count = 0

print(f"\n开始离线生成 {num_images} 张纯净测试星图 (无网格、无标注)...")

for i in range(num_images):
    # 随机生成机械转角指令
    az_deg = random.uniform(0, 360)
    alt_deg = random.uniform(15, 85)
    
    az_rad, alt_rad = math.radians(az_deg), math.radians(alt_deg)
    
    # 计算旋转矩阵
    R_J2K_ENU = rot_z(-(gmst_rad(julian_day(utc_now)) + math.radians(LON))) @ rot_y(math.radians(LAT) - math.pi/2)
    R_ENU_MNT = rot_x(EPS_X) @ rot_y(EPS_Y) @ rot_z(EPS_Z)
    R_MNT_GIM = rot_z(-az_rad) @ rot_z(THETA_NP) @ rot_x(-alt_rad) @ rot_z(-THETA_NP)
    R_total_J2K_OBS = R_J2K_ENU @ R_ENU_MNT @ R_MNT_GIM @ euler_zxy(PHI_Z, PHI_X, PHI_Y) @ rot_x(DELTA)
    
    # 获取真实天球指向
    v_j2k = R_total_J2K_OBS @ np.array([0, 0, 1])
    v_j2k /= np.linalg.norm(v_j2k)
    R_OBS_J2K = R_total_J2K_OBS.T 
    
    # 检索星星
    indices = tree.query_ball_point(v_j2k, search_radius)
    
    # 建立纯黑画布 (无任何标注线)
    star_map = np.zeros((img_height, img_width), dtype=np.float32)
    
    if indices:
        local_stars_j2k = stars_3d_all[indices]
        local_mags = mags_all[indices]
        stars_obs = (R_OBS_J2K @ local_stars_j2k.T).T
        front_mask = stars_obs[:, 2] > 0
        stars_obs, local_mags = stars_obs[front_mask], local_mags[front_mask]
        coords_homo = (K @ (stars_obs / stars_obs[:, 2:3]).T).T
        u, v = coords_homo[:, 0], coords_homo[:, 1]
        
        for j in range(len(u)):
            mag = local_mags[j]
            # 适度调高亮度系数，让亮星特征在纯黑背景下更明显
            brightness = 150.0 * (2.512 ** (6.5 - mag))
            draw_gaussian_spot(star_map, u[j], v[j], brightness, sigma=1.2)
            
    # 添加极微弱传感器底噪
    star_map += np.random.normal(3, 2, (img_height, img_width))
    final_img = np.clip(star_map, 0, 255).astype(np.uint8)
    
    # 保存图片
    # --- 修改后的文件名保存逻辑 ---
    # 文件名格式：sim_Az[角度]_Alt[角度]_F[焦距]_x[J2000x]_y[J2000y]_z[J2000z].jpg
    # 这样一张图片就包含了完整的“输入”与“结果”
    filename = f"sim_Az{az_deg:.2f}_Alt{alt_deg:.2f}_F{focal_length:.0f}_x{v_j2k[0]:.4f}_y{v_j2k[1]:.4f}_z{v_j2k[2]:.4f}.jpg"
    filepath = os.path.join(output_dir, filename)
    cv2.imwrite(filepath, final_img)
    
    print(f"[{i+1}/{num_images}] 生成成功: {filename}")
    success_count += 1

print(f"\n数据集生成完毕！请在 {output_dir} 文件夹中查看。")

正在载入全天区星表...
原数据库恒星数: 2803799，过滤后 (<= 6.5等) 剩余极亮恒星数: 11178

开始离线生成 20 张纯净测试星图 (无网格、无标注)...
[1/20] 生成成功: sim_x-0.2884_y-0.7227_z0.6282.jpg (视野亮星数: 136 颗)
[2/20] 生成成功: sim_x-0.2028_y0.0072_z0.9792.jpg (视野亮星数: 127 颗)
[3/20] 生成成功: sim_x-0.8321_y-0.1887_z0.5215.jpg (视野亮星数: 104 颗)
[4/20] 生成成功: sim_x-0.2087_y-0.9460_z-0.2480.jpg (视野亮星数: 113 颗)
[5/20] 生成成功: sim_x-0.1356_y0.0847_z0.9871.jpg (视野亮星数: 125 颗)
[6/20] 生成成功: sim_x-0.7118_y-0.6450_z0.2780.jpg (视野亮星数: 115 颗)
[7/20] 生成成功: sim_x-0.7713_y-0.3936_z-0.5002.jpg (视野亮星数: 134 颗)
[8/20] 生成成功: sim_x-0.1972_y-0.5739_z0.7949.jpg (视野亮星数: 129 颗)
[9/20] 生成成功: sim_x-0.9551_y-0.1452_z0.2582.jpg (视野亮星数: 120 颗)
[10/20] 生成成功: sim_x-0.6946_y-0.2842_z0.6609.jpg (视野亮星数: 115 颗)
[11/20] 生成成功: sim_x-0.9069_y-0.2787_z-0.3161.jpg (视野亮星数: 114 颗)
[12/20] 生成成功: sim_x-0.7649_y-0.3676_z-0.5290.jpg (视野亮星数: 142 颗)
[13/20] 生成成功: sim_x0.5821_y-0.8017_z-0.1358.jpg (视野亮星数: 140 颗)
[14/20] 生成成功: sim_x0.3044_y-0.5393_z0.7852.jpg (视野亮星数: 242 颗)
[15/20] 生成成功: sim_x-0.8612_y-0.3439